# Notebook 08 — Results Aggregation and Publication Figures

**Authors:** Syrine M.  
**Date:** 2025  
**Abstract:** Loads all saved metrics from notebooks 00-07 and produces the
final publication-ready figures and LaTeX tables for the research paper.

**Outputs:**
- Figure 1: Mapping accuracy comparison (bar chart)
- Figure 2: Routing distribution (pie chart)
- Figure 3: DQ improvement (grouped bars)
- Figure 4: Confidence vs accuracy (scatter with regression)
- Figure 5: Few-shot ablation (line chart)
- Figure 6: DVR correction ablation (line chart)
- Table 1: DQ baseline
- Table 2: Main results
- Table 3: Error analysis
- Table 4: Ablation summary

In [ ]:
# Cell 1 — Setup and load all metrics
import sys, os, json, glob
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

RESEARCH_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..' if 'notebooks' in os.getcwd() else '.'))
if 'notebooks' in os.getcwd():
    os.chdir(RESEARCH_ROOT)
sys.path.insert(0, RESEARCH_ROOT)

from src.visualizer import ResearchVisualizer
from src.evaluator import ETLEvaluator

viz = ResearchVisualizer()

# Load all metrics
metrics_dir = 'results/metrics'
all_metrics = {}
for f in sorted(glob.glob(os.path.join(metrics_dir, '*.json'))):
    key = os.path.basename(f).replace('.json', '')
    with open(f) as fh:
        all_metrics[key] = json.load(fh)
    print(f'Loaded: {f}')

print(f'\nTotal metric files: {len(all_metrics)}')
for k in all_metrics:
    print(f'  {k}: {len(all_metrics[k])} keys')

In [ ]:
# Cell 2 — Figure 1: Mapping accuracy by dataset
m06 = all_metrics.get('06_end_to_end', {})
pipeline_results = m06.get('pipeline_results', [])

if pipeline_results:
    ds_names = [r['dataset_name'] for r in pipeline_results]
    accuracies = {ds: r['mapping_accuracy'] for ds, r in zip(ds_names, pipeline_results)}
    fig = viz.plot_mapping_accuracy_by_dataset(accuracies)
    plt.show()
    print('Figure 1: Mapping accuracy by dataset')
else:
    print('No pipeline_results found. Run notebook 06 first.')

In [ ]:
# Cell 3 — Figure 2: Routing distribution
routing = m06.get('routing_distribution', {})
if routing:
    fig = viz.plot_routing_distribution(routing)
    plt.show()
    print('Figure 2: Routing distribution')
else:
    # Reconstruct from pipeline results
    if pipeline_results:
        models = [r.get('model_used', 'unknown') for r in pipeline_results]
        from collections import Counter
        routing = dict(Counter(models))
        fig = viz.plot_routing_distribution(routing)
        plt.show()
    else:
        print('No routing data available.')

In [ ]:
# Cell 4 — Figure 3: DQ improvement
m03 = all_metrics.get('03_cleaning', {})
dq_before = m03.get('dq_before', {})
dq_after = m03.get('dq_after', {})

if dq_before and dq_after:
    fig = viz.plot_dq_improvement(dq_before, dq_after)
    plt.show()
    print('Figure 3: DQ improvement')
else:
    # Use pipeline results
    if pipeline_results:
        fig, ax = plt.subplots(figsize=(8, 5))
        ds_names = [r['dataset_name'] for r in pipeline_results]
        improvements = [r.get('dq_improvement', 0) for r in pipeline_results]
        ax.bar(range(len(ds_names)), improvements, color=viz.COLORS[:len(ds_names)])
        ax.set_xticks(range(len(ds_names)))
        ax.set_xticklabels([n.replace('_', '\n') for n in ds_names], fontsize=8)
        ax.set_ylabel('DQ Improvement')
        ax.set_title('Data Quality Improvement per Dataset')
        fig.tight_layout()
        fig.savefig('results/figures/fig3_dq_improvement.pdf', dpi=300)
        plt.show()
    else:
        print('No DQ data available.')

In [ ]:
# Cell 5 — Figure 4: Confidence vs accuracy
m02 = all_metrics.get('02_schema_mapping', {})
conf_data = m02.get('confidence_vs_accuracy', {})

if conf_data:
    fig = viz.plot_confidence_vs_accuracy(
        conf_data.get('confidences', []),
        conf_data.get('accuracies', [])
    )
    plt.show()
    print('Figure 4: Confidence vs accuracy')
elif pipeline_results:
    confs = [r.get('confidence', 0.5) for r in pipeline_results]
    accs = [r.get('mapping_accuracy', 0) for r in pipeline_results]
    fig = viz.plot_confidence_vs_accuracy(confs, accs)
    plt.show()
    print('Figure 4: Confidence vs accuracy (from pipeline results)')
else:
    print('No confidence data available.')

In [ ]:
# Cell 6 — Figure 5 & 6: Ablation plots
m07 = all_metrics.get('07_ablation', {})

# Few-shot ablation
fs = m07.get('fewshot_ablation', {})
if fs:
    ds_list = sorted(set(fs.get('dataset', [])))
    k_vals = sorted(set(fs.get('k', [])))
    fs_dict = {
        ds: [fs['accuracy'][i] for i in range(len(fs['k']))
             if fs['dataset'][i] == ds]
        for ds in ds_list
    }
    fig = viz.plot_ablation_fewshot(k_vals, fs_dict)
    plt.show()
    print('Figure 5: Few-shot ablation')

# DVR correction ablation
dvr = m07.get('dvr_ablation', {})
if dvr:
    ds_list = sorted(set(dvr.get('dataset', [])))
    att_vals = sorted(set(dvr.get('max_attempts', [])))
    dvr_dict = {
        ds: [dvr['sql_valid'][i] for i in range(len(dvr['max_attempts']))
             if dvr['dataset'][i] == ds]
        for ds in ds_list
    }
    fig = viz.plot_ablation_correction(att_vals, dvr_dict)
    plt.show()
    print('Figure 6: DVR correction ablation')

In [ ]:
# Cell 7 — List all generated outputs
print('=== Generated Figures ===')
for f in sorted(glob.glob('results/figures/*.pdf')):
    print(f'  {f}')

print('\n=== Generated Tables ===')
for f in sorted(glob.glob('results/tables/*.tex')):
    print(f'  {f}')

print('\n=== Saved Metrics ===')
for f in sorted(glob.glob('results/metrics/*.json')):
    print(f'  {f}')

In [ ]:
# Cell 8 — Print summary statistics for paper
print('=' * 70)
print('SUMMARY STATISTICS FOR PAPER')
print('=' * 70)

if pipeline_results:
    accs = [r['mapping_accuracy'] for r in pipeline_results]
    recalls = [r.get('cleaning_recall', 0) for r in pipeline_results]
    dq_imps = [r.get('dq_improvement', 0) for r in pipeline_results]
    sql_rates = [1.0 if r.get('sql_valid', False) else 0.0 for r in pipeline_results]
    
    print(f"\nMapping Accuracy:")
    print(f"  Mean: {np.mean(accs):.3f} ± {np.std(accs):.3f}")
    print(f"  Min:  {np.min(accs):.3f}")
    print(f"  Max:  {np.max(accs):.3f}")
    
    print(f"\nCleaning Recall:")
    print(f"  Mean: {np.mean(recalls):.3f} ± {np.std(recalls):.3f}")
    
    print(f"\nDQ Improvement:")
    print(f"  Mean: {np.mean(dq_imps):+.2%}")
    
    print(f"\nSQL Validity Rate: {np.mean(sql_rates):.0%}")
    
    # HITL
    hitl_rate = sum(1 for r in pipeline_results if r.get('requires_human_review', False)) / len(pipeline_results)
    print(f"\nHITL Escalation Rate: {hitl_rate:.0%}")
    
    # Timing
    times = [r.get('total_time_ms', 0)/1000 for r in pipeline_results]
    print(f"\nAvg Pipeline Time: {np.mean(times):.2f}s")

print('\n✓ Notebook 08 complete. All publication materials generated.')